In [ ]:
# Import libraries - like plugging in the tools we need for the job
import pandas as pd  # For working with tabular data
import numpy as np   # For mathematical operations
import matplotlib.pyplot as plt # module for building plots and visualizations.
from sklearn.preprocessing import LabelEncoder # converts categorical (text) values into numeric codes.
import seaborn as sns # library for statistical visualization built on top of matplotlib.
from sklearn.model_selection import train_test_split  # Function for splitting data
from sklearn.tree import DecisionTreeClassifier


In [ ]:
# pd.read_csv() - reads a CSV file and creates a DataFrame. Returns a tabular data structure.
df = pd.read_csv('airline_passenger_satisfaction.csv')
# pd.set_option() - sets pandas display options. 'display.max_columns', None means show all columns without limits.
pd.set_option('display.max_columns', None)
# head() - returns the first 5 rows of the DataFrame (by default), useful for a quick look at the data structure.
df.head()


In [ ]:
# drop() - removes specified columns or rows from the DataFrame

# columns - list of columns to drop
# inplace=True - modifies the original DataFrame without creating a copy

df.drop(columns = ['id', 'Unnamed: 0'], inplace=True)
df.head()


In [ ]:
# dtypes - attribute showing the data type of each column
df.dtypes


## Data Cleaning


In [ ]:
# info() - prints a short summary of the DataFrame: data types, non-null counts, memory usage
df.info()


In [ ]:
# dropna() - removes rows that have missing values
# inplace=True - modifies the original DataFrame
df.dropna(inplace=True)


In [ ]:
df.isnull().sum()


In [ ]:
df.shape


## Charts (Data Visualization)


In [ ]:
# plt.pie() - creates a pie chart

# df['satisfaction'].value_counts() - counts the occurrences of each unique value
# labels - labels for the pie chart sectors
# autopct='%1.1f%%' - percentage display format

plt.pie(df['Gender'].value_counts(), labels=['Male', 'Female'], autopct='%1.1f%%')

# plt.show() - displays the plot on the screen
plt.show()


## Coding Data


In [ ]:
df.dtypes


In [ ]:
# LabelEncoder() - creates an encoder instance for converting text categories into numbers

label_encoder = LabelEncoder()

# df.select_dtypes(include = 'object').drop(columns='satisfaction').columns
# Select object-typed columns and drop the resulting 'satisfaction' column

columns = ['Gender', 'Type of Travel','Customer Type', 'Class'] # Apply encoding on a fixed list
for col in columns:
    df[col] = label_encoder.fit_transform(df[col])

# fit_transform() - fits the encoder and transforms the data in a single step

#  fit() - memorizes the unique category values
#  transform() - converts categories into numbers



In [ ]:
df.dtypes


## Additional Charts


In [ ]:
# sns.heatmap() - creates a correlation heatmap

# df.corr() - computes the correlation matrix
# annot=True - shows correlation values inside the cells
# fmt='.2f' - number formatting (2 digits after the decimal point)
# cmap='Greens' - color palette

plt.figure(figsize=(20,10))
sns.heatmap(df.drop(columns='satisfaction').corr(), annot=True, fmt='.2f', cmap='Greens') # Correlation plot
plt.show()


## Filtering Data


In [ ]:
# df[[list_of_columns]] - selects multiple columns from the DataFrame
df[['Gender', 'Age', 'Type of Travel']].head()


In [ ]:
# df.loc[rows, columns] - selects data by labels

# 2:5 - rows with indices from 2 to 5 inclusive
# list_of_columns - specific columns

df.loc[2:5, ['Gender', 'Age', 'Type of Travel']]


In [ ]:
# df.loc[condition, columns] - selects rows matching the condition

# df['Age']>50 - boolean condition used for filtering

df.loc[df['Age']>50, ['Gender', 'Age', 'Type of Travel']].head()


In [ ]:
df.columns # columns - attribute returns the names of all DataFrame columns


## Models (ML Algorithms)


In [ ]:
# df.drop(columns='satisfaction') - drops the 'satisfaction' column, producing the feature matrix X

X=df.drop(columns = 'satisfaction')
X.head()
X.dtypes


In [ ]:
y = df['satisfaction'] # Extracts the target variable (what we want to predict)
y.head()


In [ ]:
y.info()


### DecisionTree


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score, confusion_matrix  # Import classification metrics
from sklearn.model_selection import train_test_split # Import the function for splitting data
df_dataset = pd.read_csv('airline_passenger_satisfaction.csv')
df_dataset.info()


In [ ]:
label_encoder = LabelEncoder()

# df_dataset.select_dtypes(include='object') - select only columns whose data type is "object"
# .drop(columns='satisfaction') - drop the target variable "satisfaction",
# since we should not encode it together with the features.
# .columns - get the list of names of such text columns.
df_dataset.select_dtypes(include = 'object').drop(columns='satisfaction').columns

for col in columns:
    df_dataset[col] = label_encoder.fit_transform(df_dataset[col])


In [ ]:
df_dataset.drop(columns = ['id', 'Unnamed: 0'], inplace=True)
df_dataset.dropna(inplace=True)
df_dataset.head()


In [ ]:
X=df_dataset.drop(columns = 'satisfaction')
X.head()


In [ ]:
y = df_dataset['satisfaction']
y.head()


In [ ]:
# CORRECT split of the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                    # Our data
    test_size=0.2,          # 20% of the data for testing
    random_state=42,        # Fix randomness for reproducibility
    stratify=y              # Preserve class proportions in both sets
)


In [ ]:
model = DecisionTreeClassifier() # DecisionTreeClassifier() - creates a decision tree model with default parameters
model.fit(X_train, y_train) # fit() - trains the model on X (features) and y (target values)


In [ ]:
predictions = model.predict(X_test) # predict() - produces predictions for new data X_test


In [ ]:
# Compute all model quality metrics
accuracy = accuracy_score(y_test, predictions)  # accuracy_score() - compares true y_test with predictions and returns the fraction of matches
# pos_label='satisfied' -> compute the metric for a single class only. Only for binary classification (satisfied vs neutral or dissatisfied).
# In this mode the metric answers: "How well does the model find the 'satisfied' class in particular?"
precision = precision_score(y_test, predictions, pos_label='satisfied')  # precision_score() - computes precision (of the predicted positives, how many are truly positive)
recall = recall_score(y_test, predictions, pos_label='satisfied')  # recall_score() - computes recall (of all actual positives, how many we found)
f1 = f1_score(y_test, predictions, pos_label='satisfied')  # f1_score() - harmonic mean of precision and recall

# average='weighted' -> take both classes and average their results, weighted by class counts.
# average='weighted' answers: "On average across all classes, how well does the model perform"
f_beta_05 = fbeta_score(y_test, predictions, beta=0.5, average='weighted')  # fbeta_score() with beta=0.5 - more weight on precision
f_beta_2 = fbeta_score(y_test, predictions, beta=2.0, average='weighted')  # fbeta_score() with beta=2.0 - more weight on recall

print("📊 DECISION TREE MODEL EVALUATION RESULTS:")
print("="*50)
print(f"Accuracy:                         {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision (positive class):       {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:                           {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score:                         {f1:.4f} ({f1*100:.2f}%)")
print(f"F-Beta (beta=0.5, favors Precision): {f_beta_05:.4f} ({f_beta_05*100:.2f}%)")
print(f"F-Beta (beta=2.0, favors Recall):    {f_beta_2:.4f} ({f_beta_2*100:.2f}%)")

print(f"\n📝 Interpretation of the results:")
print(f"• Out of {len(y_test)} passengers, the model correctly classified {int(accuracy * len(y_test))}")
print(f"• Overall accuracy is {accuracy*100:.2f}%")


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, predictions)
print(f"\n🔍 CONFUSION MATRIX:")
print(f"True Negative (correctly 'dissatisfied'): {cm[0,0]}")
print(f"False Positive (wrongly 'satisfied'): {cm[0,1]}")
print(f"False Negative (wrongly 'dissatisfied'): {cm[1,0]}")
print(f"True Positive (correctly 'satisfied'): {cm[1,1]}")


### Random Forest (Homework)


In [ ]:
from sklearn.ensemble import RandomForestClassifier # RandomForestClassifier - the "random forest" algorithm, which combines many decision trees

model = RandomForestClassifier()
model.fit(X, y)

predictions = model.predict(X_test)
predictions

accuracy_score(y_test, predictions)


### KNeighborsClassifier (Homework)


In [ ]:
X_test.isnull().sum()


In [ ]:
from sklearn.neighbors import KNeighborsClassifier # KNeighborsClassifier - k-nearest neighbors algorithm; n_neighbors=5 - number of neighbors used for classification

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X, y)

predictions = model.predict(X_test)
predictions

accuracy_score(y_test, predictions)


### Logistic Regression (Homework)


In [ ]:
from sklearn.linear_model import LogisticRegression # LogisticRegression - logistic regression algorithm; max_iter=10000 - max number of iterations for convergence

model = LogisticRegression(max_iter=10000)
model.fit(X, y)

predictions = model.predict(X_test)
predictions

accuracy_score(y_test, predictions)


In [ ]:
model.predict(new_df)


## Saving prediction model


In [ ]:
import joblib


In [ ]:
# joblib.dump() - saves a model to a file

# model - model object to save
# 'filename.joblib' - target filename

joblib.dump(model, 'airl_pass_satisf.joblib')


In [ ]:
test_inputs = {
 'Gender': [1, 0],
 'Customer Type': [0, 1],
 'Age': [25, 15],
 'Type of Travel': [0, 1],
 'Class': [1, 0],
 'Flight Distance': [800, 2000],
 'Departure Delay in Minutes': [0, 10],
 'Arrival Delay in Minutes': [0, 20]
}

test_inputs


In [ ]:
trained_model = joblib.load('airl_pass_satisf.joblib') # joblib.load() - loads a saved model from a file


In [ ]:
new_df = pd.DataFrame(test_inputs)
new_df

trained_model.predict(new_df)
